# Assigment 2 Machine Learning
## 2. Predict passenger data for Ruter.

# TITLE : Ruter Passenger prediction

We will predict the nummber of passengers for Ruter bus line 100 based on a specific date.

This use case uses regression because the target variable, "passasjerer_omborrd" is a numerical value.

In [ ]:
%pip install scikit-learn

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.metrics import accuracy_score, confusion_matrix

Loading the ruter dataset.

In [17]:
df = pd.read_csv("data/Ruter-data.csv", sep=";")
df.head()


,TurId,Dato,Fylke,Område,Kommune,Holdeplass_Fra,Holdeplass_Til,Linjetype,Linjefylke,Linjenavn,Linjeretning,Tidspunkt_Faktisk_Ankomst_Holdeplass_Fra,Tidspunkt_Faktisk_Avgang_Holdeplass_Fra,Tidspunkt_Planlagt_Ankomst_Holdeplass_Fra,Tidspunkt_Planlagt_Avgang_Holdeplass_Fra,Kjøretøy_Kapasitet,Passasjerer_Ombord
0,15006-2020-08-10T10:24:00+02:00,10/08/2020,Viken,Vest,Bærum,Nordliveien,Tjernsmyr,Lokal,Viken,150,0,10:53:53,10:53:59,10:53:00,10:53:00,112,5
1,15002-2020-08-15T12:54:00+02:00,15/08/2020,Viken,Vest,Bærum,Nadderud stadion,Bekkestua bussterminal (Plattform C),Lokal,Viken,150,0,13:12:20,13:12:26,13:12:00,13:12:00,112,5
2,15004-2020-08-03T09:54:00+02:00,03/08/2020,Viken,Vest,Bærum,Ringstabekkveien,Skallum,Lokal,Viken,150,0,10:18:56,10:19:21,10:19:00,10:19:00,112,6
3,15003-2020-07-27T13:00:00+02:00,27/07/2020,Viken,Vest,Bærum,Gruvemyra,Gullhaug,Lokal,Viken,150,1,13:52:04,13:52:26,13:51:00,13:51:00,112,10
4,15002-2020-08-27T07:15:00+02:00,27/08/2020,Viken,Vest,Bærum,Lysaker stasjon (Plattform A),Tjernsmyr,Lokal,Viken,150,1,07:34:13,07:34:53,07:33:00,07:33:00,112,10


Looking through the data


In [25]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

Number of rows: 6000
Number of columns: 17

Columns:
['TurId', 'Dato', 'Fylke', 'Område', 'Kommune', 'Holdeplass_Fra', 'Holdeplass_Til', 'Linjetype', 'Linjefylke', 'Linjenavn', 'Linjeretning', 'Tidspunkt_Faktisk_Ankomst_Holdeplass_Fra', 'Tidspunkt_Faktisk_Avgang_Holdeplass_Fra', 'Tidspunkt_Planlagt_Ankomst_Holdeplass_Fra', 'Tidspunkt_Planlagt_Avgang_Holdeplass_Fra', 'Kjøretøy_Kapasitet', 'Passasjerer_Ombord']


Selected bus line 100.

In [24]:
bus_data = df[df["Linjenavn"].astype(str) == "100"].copy()

print("Number of records for bus line 100:", len(bus_data))

bus_data.head()

Number of records for bus line 100: 465


,TurId,Dato,Fylke,Område,Kommune,Holdeplass_Fra,Holdeplass_Til,Linjetype,Linjefylke,Linjenavn,Linjeretning,Tidspunkt_Faktisk_Ankomst_Holdeplass_Fra,Tidspunkt_Faktisk_Avgang_Holdeplass_Fra,Tidspunkt_Planlagt_Ankomst_Holdeplass_Fra,Tidspunkt_Planlagt_Avgang_Holdeplass_Fra,Kjøretøy_Kapasitet,Passasjerer_Ombord
38,10008-2020-08-09T11:24:00+02:00,09/08/2020,Viken,Nordøst,Lillestrøm,Sagdalen,Sagdalen skole,Lokal,Viken,100,0,11:38:58,11:39:06,11:38:00,11:38:00,151,1
42,10004-2020-08-18T04:59:00+02:00,18/08/2020,Oslo,Nordøst,Alna,IKEA Furuset (mot Kjelsrud),Kjelsrud (fra Furuset),Lokal,Viken,100,0,05:42:09,05:42:18,05:42:00,05:42:00,151,0
43,10009-2020-07-21T01:00:00+02:00,20/07/2020,Viken,Nordøst,Lørenskog,Vittenberg,Strømsbergveien,Lokal,Viken,100,1,25:45:48,25:45:54,25:40:00,25:40:00,151,7
46,10004-2020-07-04T18:24:00+02:00,04/07/2020,Viken,Nordøst,Lillestrøm,UTGÅTT Lillestrøm bussterminal (P 19a),Jonas Lies gate,Lokal,Viken,100,0,18:34:37,18:35:32,18:33:00,18:33:00,151,13
51,10002-2020-07-22T08:30:00+02:00,22/07/2020,Oslo,Indre By,Gamle Oslo,Oslo bussterminal (Plf B10),Oslo gate (mot Galgeberg),Lokal,Viken,100,1,08:15:00,08:25:43,08:30:00,08:30:00,151,0


Preparing the data. 
The data is stored as text, the code converts it to numericsl.

In [23]:
bus_data["Dato"] = pd.to_datetime(
    bus_data["Dato"],
    dayfirst=True
)

bus_data.head()

,TurId,Dato,Fylke,Område,Kommune,Holdeplass_Fra,Holdeplass_Til,Linjetype,Linjefylke,Linjenavn,Linjeretning,Tidspunkt_Faktisk_Ankomst_Holdeplass_Fra,Tidspunkt_Faktisk_Avgang_Holdeplass_Fra,Tidspunkt_Planlagt_Ankomst_Holdeplass_Fra,Tidspunkt_Planlagt_Avgang_Holdeplass_Fra,Kjøretøy_Kapasitet,Passasjerer_Ombord
38,10008-2020-08-09T11:24:00+02:00,2020-08-09,Viken,Nordøst,Lillestrøm,Sagdalen,Sagdalen skole,Lokal,Viken,100,0,11:38:58,11:39:06,11:38:00,11:38:00,151,1
42,10004-2020-08-18T04:59:00+02:00,2020-08-18,Oslo,Nordøst,Alna,IKEA Furuset (mot Kjelsrud),Kjelsrud (fra Furuset),Lokal,Viken,100,0,05:42:09,05:42:18,05:42:00,05:42:00,151,0
43,10009-2020-07-21T01:00:00+02:00,2020-07-20,Viken,Nordøst,Lørenskog,Vittenberg,Strømsbergveien,Lokal,Viken,100,1,25:45:48,25:45:54,25:40:00,25:40:00,151,7
46,10004-2020-07-04T18:24:00+02:00,2020-07-04,Viken,Nordøst,Lillestrøm,UTGÅTT Lillestrøm bussterminal (P 19a),Jonas Lies gate,Lokal,Viken,100,0,18:34:37,18:35:32,18:33:00,18:33:00,151,13
51,10002-2020-07-22T08:30:00+02:00,2020-07-22,Oslo,Indre By,Gamle Oslo,Oslo bussterminal (Plf B10),Oslo gate (mot Galgeberg),Lokal,Viken,100,1,08:15:00,08:25:43,08:30:00,08:30:00,151,0


Combine data to find the average passenger count for bus line 100 on each date.
This is because there are several observations for bus line 100 on one day ,because the bus stops at many different stops.

In [22]:
daily_data = bus_data.groupby("Dato")["Passasjerer_Ombord"].mean().reset_index()

daily_data.head()

,Dato,Passasjerer_Ombord
0,2020-06-01,5.666667
1,2020-06-02,6.333333
2,2020-06-03,10.000000
3,2020-06-04,6.333333
4,2020-06-05,1.500000


Converting the date format to number,
  this makes it easier for the machine model

In [21]:
daily_data["date_number"] = (
    daily_data["Dato"] - pd.Timestamp("2020-01-01")
).dt.days

daily_data.head()

,Dato,Passasjerer_Ombord,date_number
0,2020-06-01,5.666667,152
1,2020-06-02,6.333333,153
2,2020-06-03,10.000000,154
3,2020-06-04,6.333333,155
4,2020-06-05,1.500000,156


# Research on the Algorithm choice

This is a regression problem because the number of passengers is a continuous numerical value and not fixed.
I chose Linear Regression because it is simple and suitable for predicting numerical values.
The model tries to find a relationship between the date and the number of passengers.


Creating var X (input = date) and var y (output = passangers). 

In [20]:
X = daily_data[["date_number"]]

y = daily_data["Passasjerer_Ombord"]

print("X:")
print(X.head())

print("\ny:")
print(y.head())

X:
   date_number
0          152
1          153
2          154
3          155
4          156

y:
0     5.666667
1     6.333333
2    10.000000
3     6.333333
4     1.500000
Name: Passasjerer_Ombord, dtype: float64


Spliting the data into training and testing sets:

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 72
Testing data: 18


Training the Model


In [18]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


Make Prediction

Accuracy and evaluation

Predict a specific date
